In [1]:
import tensorflow as tf
from tensorflow.keras.layers import Input,SimpleRNN,Dense,Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam,SGD
import numpy as np
import matplotlib.pyplot as plt

2026-02-01 12:44:52.561753: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-01 12:44:52.596105: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-01 12:44:53.561990: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
N=1
T=10
D=3
K=2
X=np.random.randn(N,T,D)


In [3]:
M=5
i=Input(shape=(T,D))
x=SimpleRNN(M)(i)
x=Dense(K)(x)
model=Model(i,x)

I0000 00:00:1769931894.803563 1468704 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3843 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1660 SUPER, pci bus id: 0000:01:00.0, compute capability: 7.5


In [7]:
Yhat=model.predict(X)
print("Output:", Yhat)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
Output: [[-0.5532768 -1.020731 ]]


In [8]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 10, 3)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 5)              │            45 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │            12 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 57 (228.00 B)

 Trainable params: 57 (228.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
model.layers[1].get_weights()

[array([[-0.49956235, -0.69394994, -0.81907964,  0.7104172 ,  0.00156611],
        [ 0.08891332,  0.5770485 ,  0.71597534, -0.70494395, -0.38600975],
        [ 0.00091201, -0.7891418 ,  0.7901775 , -0.5766386 ,  0.38197643]],
       dtype=float32),
 array([[ 0.47879326, -0.35598445, -0.2212416 , -0.54656   ,  0.54438627],
        [ 0.86542547,  0.15267618,  0.09573318,  0.4242023 , -0.19651043],
        [ 0.09082379, -0.19867879, -0.4821973 , -0.3606896 , -0.7678976 ],
        [ 0.11636337,  0.46782607,  0.58764416, -0.6233856 , -0.18347536],
        [-0.00249628, -0.76917535,  0.6033636 ,  0.05116871, -0.20419925]],
       dtype=float32),
 array([0., 0., 0., 0., 0.], dtype=float32)]

In [9]:
a,b,c=model.layers[1].get_weights()
print(a.shape,b.shape,c.shape)

(3, 5) (5, 5) (5,)


In [11]:
Wx,Wh,bh=model.layers[1].get_weights()
Wo,bo=model.layers[2].get_weights()

In [12]:
h_last = np.zeros(M)  # initial hidden state
x = X[0]  # the one and only sample
Yhats = []  # where we store the outputs

for t in range(T):
  h = np.tanh(x[t].dot(Wx) + h_last.dot(Wh) + bh)
  y = h.dot(Wo) + bo  # we only care about this value on the last iteration
  Yhats.append(y)

  # important: assign h to h_last
  h_last = h

# print the final output
print(Yhats[-1])

[-0.55327673 -1.02073105]
